# 📊 Data Exploration & Quality Analysis
## ERP Sales Analytics - Understanding the Shoebadoo E-Commerce Data

**Objective:** 
- Load and understand raw data structure
- Assess data quality (null values, duplicates, data types)
- Identify data quality issues
- Generate initial statistics and insights
- Document findings for cleaning pipeline

## 1. Setup & Imports

In [1]:
# Imports
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when, sum as spark_sum, avg, min as spark_min, max as spark_max
from pyspark.sql.types import *
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

print("✅ Imports erfolgreich!")

✅ Imports erfolgreich!


## 2. Spark Session erstellen
Da ich nicht mehr mit Databricks arbeite sondern im Docker lokal, muss ich eine Sparksession starten.

In [2]:
# Spark Session erstellen (ersetzt den vorhandenen 'spark' in Databricks)
spark = SparkSession.builder \
    .appName("ERP-Sales-Data-Exploration") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .config("spark.driver.memory", "2g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

# Spark Version anzeigen
print(f"✅ Spark Session erstellt!")
print(f"   Spark Version: {spark.version}")
print(f"   App Name: {spark.sparkContext.appName}")

✅ Spark Session erstellt!
   Spark Version: 3.5.0
   App Name: ERP-Sales-Data-Exploration


## 3. Daten laden

**Hinweis:** Wir laden die Daten zuerst via Pandas (wie in Databricks), um INT64-Timestamp-Probleme zu umgehen.
Ich habe hier nurn DATA_PATH reingetan, um mit den Paths flexibler zu sein.

In [5]:
# Pfad-Konfiguration (Docker Container Pfad)
DATA_PATH = r"C:\Users\Apfel\Documents\bewerbungen\data analyst\ERP-sales-analytics\erp-sales-analytics-pipeline\data\raw"

print("📂 Lade Daten aus:", DATA_PATH)
print("-" * 80)

# Via Pandas laden (umgeht Spark Parquet INT64-Timestamp Probleme)
try:
    customers_pandas = pd.read_parquet(f"{DATA_PATH}/customers.parquet")
    print("✅ customers.parquet geladen")
    
    products_pandas = pd.read_parquet(f"{DATA_PATH}/products.parquet")
    print("✅ products.parquet geladen")
    
    returns_pandas = pd.read_parquet(f"{DATA_PATH}/returns.parquet")
    print("✅ returns.parquet geladen")
    
    sales_pandas = pd.read_parquet(f"{DATA_PATH}/sales.parquet")
    print("✅ sales.parquet geladen")
    
    print("\n🎉 Alle Dateien erfolgreich geladen!")
    
except Exception as e:
    print(f"❌ Fehler beim Laden: {e}")
    print("\n💡 Tipp: Prüfe ob die Parquet-Dateien in /app/data/raw/ liegen")

📂 Lade Daten aus: C:\Users\Apfel\Documents\bewerbungen\data analyst\ERP-sales-analytics\erp-sales-analytics-pipeline\data\raw
--------------------------------------------------------------------------------
✅ customers.parquet geladen
✅ products.parquet geladen
✅ returns.parquet geladen
✅ sales.parquet geladen

🎉 Alle Dateien erfolgreich geladen!


## 4. Zu Spark DataFrames konvertieren

In [6]:
# Pandas → Spark DataFrames
print("🔄 Konvertiere zu Spark DataFrames...\n")

customers_df = spark.createDataFrame(customers_pandas)
print(f"✅ customers_df: {customers_df.count():,} rows")

products_df = spark.createDataFrame(products_pandas)
print(f"✅ products_df:  {products_df.count():,} rows")

returns_df = spark.createDataFrame(returns_pandas)
print(f"✅ returns_df:   {returns_df.count():,} rows")

sales_df = spark.createDataFrame(sales_pandas)
print(f"✅ sales_df:     {sales_df.count():,} rows")

print("\n🎉 Konvertierung abgeschlossen!")

🔄 Konvertiere zu Spark DataFrames...



Py4JJavaError: An error occurred while calling o54.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Task 11 in stage 0.0 failed 1 times, most recent failure: Lost task 11.0 in stage 0.0 (TID 11) (host.docker.internal executor driver): java.io.IOException: Cannot run program "python3": CreateProcess error=2, Das System kann die angegebene Datei nicht finden
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1143)
	at java.base/java.lang.ProcessBuilder.start(ProcessBuilder.java:1073)
	at org.apache.spark.api.python.PythonWorkerFactory.createSimpleWorker(PythonWorkerFactory.scala:181)
	at org.apache.spark.api.python.PythonWorkerFactory.create(PythonWorkerFactory.scala:109)
	at org.apache.spark.SparkEnv.createPythonWorker(SparkEnv.scala:124)
	at org.apache.spark.api.python.BasePythonRunner.compute(PythonRunner.scala:174)
	at org.apache.spark.api.python.PythonRDD.compute(PythonRDD.scala:67)
	at org.apache.spark.rdd.RDD.computeOrReadCheckpoint

## 5. Schemas inspizieren

In [7]:
print("\n" + "="*80)
print(" CUSTOMERS SCHEMA")
print("="*80)
customers_df.printSchema()


 CUSTOMERS SCHEMA
root
 |-- customer_id: long (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- registration_date: date (nullable = true)
 |-- country: string (nullable = true)
 |-- date_of_birth: date (nullable = true)



In [8]:
print("\n" + "="*80)
print(" PRODUCTS SCHEMA")
print("="*80)
products_df.printSchema()


 PRODUCTS SCHEMA


NameError: name 'products_df' is not defined

In [ ]:
print("\n" + "="*80)
print(" SALES SCHEMA")
print("="*80)
sales_df.printSchema()

In [ ]:
print("\n" + "="*80)
print(" RETURNS SCHEMA")
print("="*80)
returns_df.printSchema()

## 6. Data Quality Report Funktion

**Verbesserte Version** deiner Funktion mit zusätzlichen Features

In [ ]:
def quality_report(df, dataset_name):
    """
    Erstellt einen ausführlichen Data Quality Report.
    
    Args:
        df: PySpark DataFrame
        dataset_name: Name des Datasets für die Ausgabe
    
    Returns:
        dict: Zusammenfassung der Quality Metrics
    """
    print(f"\n{'='*80}")
    print(f" 📊 {dataset_name.upper()}")
    print(f"{'='*80}")
    
    # Basis Statistiken
    total = df.count()
    distinct = df.distinct().count()
    duplicates = total - distinct
    
    print(f"\n📈 Überblick:")
    print(f"   Total Rows:     {total:,}")
    print(f"   Distinct Rows:  {distinct:,}")
    print(f"   Duplicates:     {duplicates:,} ({duplicates/total*100:.2f}%)")
    print(f"   Columns:        {len(df.columns)}")
    
    # Null Values pro Spalte
    print(f"\n🔍 Null Values:")
    null_info = {}
    
    for col_name in df.columns:
        null_count = df.filter(col(col_name).isNull()).count()
        null_info[col_name] = null_count
        
        if null_count > 0:
            percentage = null_count / total * 100
            status = "⚠️" if percentage > 10 else "⚡"
            print(f"   {status} {col_name:30s}: {null_count:>6,} ({percentage:>5.1f}%)")
        else:
            print(f"   ✅ {col_name:30s}: {null_count:>6,}")
    
    # Spalten-Datentypen
    print(f"\n🏷️ Datentypen:")
    for field in df.schema.fields:
        print(f"   {field.name:30s}: {field.dataType}")
    
    # Sample Data
    print(f"\n📋 First 3 rows:")
    df.limit(3).show(truncate=False)
    
    # Rückgabe für spätere Analyse
    return {
        'dataset': dataset_name,
        'total_rows': total,
        'distinct_rows': distinct,
        'duplicates': duplicates,
        'columns': len(df.columns),
        'null_info': null_info
    }

print("✅ Quality Report Funktion definiert!")

## 7. Quality Reports ausführen

In [ ]:
# Quality Reports für alle Datasets erstellen
reports = []

reports.append(quality_report(customers_df, "CUSTOMERS"))
reports.append(quality_report(products_df, "PRODUCTS"))
reports.append(quality_report(sales_df, "SALES"))
reports.append(quality_report(returns_df, "RETURNS"))

## 8. Zusammenfassender Report

In [ ]:
print("\n" + "="*80)
print(" 📊 ZUSAMMENFASSUNG - DATA QUALITY REPORT")
print("="*80 + "\n")

for report in reports:
    print(f"Dataset: {report['dataset']}")
    print(f"  Total Rows:    {report['total_rows']:,}")
    print(f"  Distinct Rows: {report['distinct_rows']:,}")
    print(f"  Duplicates:    {report['duplicates']:,}")
    print(f"  Columns:       {report['columns']}")
    
    # Anzahl Spalten mit Nulls
    cols_with_nulls = sum(1 for count in report['null_info'].values() if count > 0)
    print(f"  Null Columns:  {cols_with_nulls}/{report['columns']}")
    print()

print("="*80)

## 9. Identifizierte Probleme dokumentieren

Basierend auf den Quality Reports oben:

In [ ]:
print("\n" + "="*80)
print(" 🚨 IDENTIFIZIERTE DATENQUALITÄTSPROBLEME")
print("="*80 + "\n")

problems = []

# Duplikate
for report in reports:
    if report['duplicates'] > 0:
        problems.append(f"❌ {report['dataset']}: {report['duplicates']:,} Duplikate gefunden")

# Null Values
for report in reports:
    for col_name, null_count in report['null_info'].items():
        if null_count > 0:
            percentage = null_count / report['total_rows'] * 100
            if percentage > 10:  # Nur wichtige Probleme
                problems.append(f"⚠️ {report['dataset']}.{col_name}: {percentage:.1f}% NULL")

# Ausgabe
if problems:
    for i, problem in enumerate(problems, 1):
        print(f"{i}. {problem}")
else:
    print("✅ Keine kritischen Probleme gefunden!")

print("\n" + "="*80)

## 10. Speichern als bereinigte Parquet (optional)

**Hinweis:** In Docker speichern wir lokal statt in Unity Catalog

In [ ]:
# Optional: Als Parquet speichern für spätere Nutzung
OUTPUT_PATH = "/app/data/cleaned"

print(f"\n💾 Speichere Daten nach: {OUTPUT_PATH}")
print("-" * 80)

# Speichern (falls du die rohen Daten erstmal so lassen willst)
try:
    customers_df.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/customers_raw.parquet")
    print("✅ customers_raw.parquet gespeichert")
    
    products_df.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/products_raw.parquet")
    print("✅ products_raw.parquet gespeichert")
    
    sales_df.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/sales_raw.parquet")
    print("✅ sales_raw.parquet gespeichert")
    
    returns_df.write.mode("overwrite").parquet(f"{OUTPUT_PATH}/returns_raw.parquet")
    print("✅ returns_raw.parquet gespeichert")
    
    print("\n🎉 Alle Dateien erfolgreich gespeichert!")
    
except Exception as e:
    print(f"⚠️ Fehler beim Speichern: {e}")

## 11. Next Steps

Based on this data exploration:

1. ✅ **Data Cleaning** → `02_data_cleaning.ipynb`
   - Remove duplicates
   - Handle missing values intelligently
   - NLP: Extract category/brand from product descriptions
   - Correct data types

2. ✅ **Data Quality Validation** → `03_data_quality_validation.ipynb`
   - Great Expectations framework setup
   - Define data contracts (JSON Schema)
   - Automated validation pipeline

3. ✅ **Dimensional Modeling** → `04_dimensional_modeling.ipynb`
   - Design star schema (Kimball methodology)
   - Build fact & dimension tables
   - Delta Lake integration

4. ✅ **Analytics Dashboard**
   - Streamlit multi-page application
   - Interactive visualizations with Plotly
   - KPI monitoring

In [ ]:
# Spark Session beenden
spark.stop()
print("\n✅ Spark Session stopped. Data Exploration complete!")